# NVDA Stock Analysis - Dashboard Data Pipeline

## Objetivo
Este notebook prepara y enriquece los datos de `gold_nvda_features` para alimentar el dashboard de análisis NVDA.

## Flujo de Datos
```
gold_nvda_features (UC Table)
    ↓
Enriquecimiento (8 columnas derivadas)
    ↓
Vista SQL: nvda_dashboard_data
    ↓
Dashboard Visualizations
```

## Columnas Enriquecidas
1. **Tendencia**: "Sube" / "Baja" (basado en target)
2. **daily_return**: Retorno porcentual diario
3. **precio_sobre_sma7**: "Alcista" / "Bajista"
4. **precio_sobre_sma30**: "Alcista" / "Bajista"
5. **volatilidad_categoria**: "Baja" / "Media" / "Alta"
6. **volumen_millones**: Volumen en millones
7. **distancia_sma7_pct**: Distancia porcentual a SMA 7
8. **distancia_sma30_pct**: Distancia porcentual a SMA 30

In [0]:
# Cargar datos de la capa Gold
from pyspark.sql import functions as F

df = spark.table("workspace.default.gold_nvda_features")

# Validaciones básicas
print(f"✓ Total de registros: {df.count():,}")
print(f"✓ Rango de fechas: {df.agg(F.min('date')).collect()[0][0]} → {df.agg(F.max('date')).collect()[0][0]}")
print(f"\n✓ Schema:")
df.printSchema()
print(f"\n✓ Valores nulos por columna:")
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()
print(f"\n✓ Primeros 5 registros:")
display(df.orderBy(F.desc("date")).limit(5))

In [0]:
# Enriquecer DataFrame con columnas derivadas
from pyspark.sql import functions as F

df_enriched = df.withColumn(
    "Tendencia",
    F.when(F.col("target") == 1, "Sube").otherwise("Baja")
).withColumn(
    "daily_return",
    ((F.col("close_price") - F.col("open_price")) / F.col("open_price") * 100)
).withColumn(
    "precio_sobre_sma7",
    F.when(F.col("close_price") > F.col("sma_7"), "Alcista").otherwise("Bajista")
).withColumn(
    "precio_sobre_sma30",
    F.when(F.col("close_price") > F.col("sma_30"), "Alcista").otherwise("Bajista")
).withColumn(
    "volatilidad_categoria",
    F.when(F.col("daily_volatility") < 2, "Baja")
     .when(F.col("daily_volatility").between(2, 5), "Media")
     .otherwise("Alta")
).withColumn(
    "volumen_millones",
    F.col("volume") / 1000000
).withColumn(
    "distancia_sma7_pct",
    ((F.col("close_price") - F.col("sma_7")) / F.col("sma_7") * 100)
).withColumn(
    "distancia_sma30_pct",
    ((F.col("close_price") - F.col("sma_30")) / F.col("sma_30") * 100)
)

print(f"✓ DataFrame enriquecido con {len(df_enriched.columns)} columnas totales")
print(f"✓ Nuevas columnas agregadas: {len(df_enriched.columns) - len(df.columns)}")
print(f"\n✓ Muestra de columnas enriquecidas:")
display(df_enriched.select("date", "close_price", "Tendencia", "daily_return", "precio_sobre_sma7", "volatilidad_categoria").orderBy(F.desc("date")).limit(5))

In [0]:
# Calcular métricas agregadas para análisis
from pyspark.sql import functions as F

# Métricas generales
total_dias = df_enriched.count()
dias_alcistas = df_enriched.filter(F.col("Tendencia") == "Sube").count()
dias_bajistas = df_enriched.filter(F.col("Tendencia") == "Baja").count()
pct_alcistas = (dias_alcistas / total_dias) * 100
pct_bajistas = (dias_bajistas / total_dias) * 100
volatilidad_promedio = df_enriched.agg(F.avg("daily_volatility")).collect()[0][0]

print("=" * 60)
print("MÉTRICAS GENERALES DEL DATASET")
print("=" * 60)
print(f"Total de días analizados: {total_dias:,}")
print(f"\nDistribución de tendencias:")
print(f"  • Días alcistas (Sube): {dias_alcistas:,} ({pct_alcistas:.1f}%)")
print(f"  • Días bajistas (Baja): {dias_bajistas:,} ({pct_bajistas:.1f}%)")
print(f"\nVolatilidad promedio: {volatilidad_promedio:.2f}%")

# Agregaciones mensuales
print(f"\n{'='*60}")
print("DISTRIBUCIÓN MENSUAL")
print("=" * 60)
df_monthly = df_enriched.withColumn("year_month", F.date_format("date", "yyyy-MM")) \
    .groupBy("year_month") \
    .agg(
        F.count("*").alias("total_dias"),
        F.sum(F.when(F.col("Tendencia") == "Sube", 1).otherwise(0)).alias("dias_alcistas"),
        F.avg("daily_volatility").alias("volatilidad_promedio")
    ).orderBy(F.desc("year_month"))

display(df_monthly.limit(12))

# Distribución de categorías de volatilidad
print(f"\n{'='*60}")
print("DISTRIBUCIÓN DE VOLATILIDAD")
print("=" * 60)
df_vol_dist = df_enriched.groupBy("volatilidad_categoria").count().orderBy("volatilidad_categoria")
display(df_vol_dist)

In [0]:
# Crear vista temporal principal para el dashboard
df_enriched.createOrReplaceTempView("nvda_dashboard_data")

print("✓ Vista temporal 'nvda_dashboard_data' creada exitosamente")
print(f"✓ Registros disponibles: {spark.table('nvda_dashboard_data').count():,}")

# Test query
print("\n✓ Test query - Últimos 5 registros:")
test_df = spark.sql("""
    SELECT 
        date,
        close_price,
        Tendencia,
        daily_return,
        precio_sobre_sma7,
        volatilidad_categoria,
        volumen_millones
    FROM nvda_dashboard_data
    ORDER BY date DESC
    LIMIT 5
""")
display(test_df)

In [0]:
# Crear vistas adicionales para análisis específicos

# Vista 1: Resumen diario
spark.sql("""
    CREATE OR REPLACE TEMP VIEW nvda_daily_summary AS
    SELECT 
        date,
        close_price,
        daily_return,
        volume,
        daily_volatility,
        Tendencia,
        volatilidad_categoria
    FROM nvda_dashboard_data
    ORDER BY date DESC
""")

# Vista 2: Análisis de tendencias
spark.sql("""
    CREATE OR REPLACE TEMP VIEW nvda_trend_analysis AS
    SELECT 
        date,
        close_price,
        sma_7,
        sma_30,
        precio_sobre_sma7,
        precio_sobre_sma30,
        distancia_sma7_pct,
        distancia_sma30_pct
    FROM nvda_dashboard_data
    ORDER BY date DESC
""")

# Vista 3: Métricas de rendimiento
spark.sql("""
    CREATE OR REPLACE TEMP VIEW nvda_performance_metrics AS
    SELECT 
        date,
        daily_return,
        daily_volatility,
        volatilidad_categoria,
        volumen_millones,
        Tendencia
    FROM nvda_dashboard_data
    ORDER BY date DESC
""")

print("✓ Vistas adicionales creadas:")
print("  • nvda_daily_summary")
print("  • nvda_trend_analysis")
print("  • nvda_performance_metrics")

In [0]:
# Verificar todas las vistas y mostrar muestras

print("=" * 70)
print("VERIFICACIÓN DE VISTAS TEMPORALES")
print("=" * 70)

views = ["nvda_dashboard_data", "nvda_daily_summary", "nvda_trend_analysis", "nvda_performance_metrics"]

for view in views:
    count = spark.table(view).count()
    print(f"\n✓ Vista: {view}")
    print(f"  Registros: {count:,}")
    print(f"  Columnas: {len(spark.table(view).columns)}")

print(f"\n{'='*70}")
print("MUESTRA DE DATOS - nvda_dashboard_data (Top 3)")
print("=" * 70)
display(spark.table("nvda_dashboard_data").orderBy(F.desc("date")).limit(3))

print(f"\n{'='*70}")
print("✅ PIPELINE COMPLETADO EXITOSAMENTE")
print("=" * 70)
print("\nTodas las vistas están listas para alimentar el dashboard.")
print("Los widgets pueden consumir los datos desde 'nvda_dashboard_data'.")
print("\nNota: Las vistas temporales persisten durante la sesión de Spark.")
print("      Se recrean automáticamente cuando el job ejecuta este notebook.")

In [0]:
# Persistir datos enriquecidos como tabla física en Unity Catalog
from pyspark.sql import functions as F

table_name = "workspace.default.gold_nvda_dashboard"

print(f"⌛ Guardando datos enriquecidos en tabla UC: {table_name}")

# Escribir como tabla Delta con sobrescritura
df_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"\n✅ Tabla '{table_name}' creada exitosamente")

# Verificar la tabla
verify_df = spark.table(table_name)
record_count = verify_df.count()
column_count = len(verify_df.columns)

print(f"\n✓ Verificación:")
print(f"  Registros: {record_count:,}")
print(f"  Columnas: {column_count}")
print(f"\n✓ Muestra de 3 registros más recientes:")
display(verify_df.orderBy(F.desc("date")).limit(3))

print(f"\n{'='*70}")
print("✅ DATOS PERSISTIDOS - Dashboard puede leer de la tabla UC")
print("=" * 70)
print(f"\nEl dashboard debe configurarse para leer desde:")
print(f"  ➡️  {table_name}")
print(f"\nColumnas disponibles: {', '.join(verify_df.columns[:10])}...")